# ResNet_from_scratch (blank)

- 빈칸을 주석, 치트시트, 쿡북을 활용해 채워보시기 바랍니다.
- 빈칸 옆 주석은 **역할과 의도** 중심으로 작성되었습니다.
- 아래 **실습 코드** 는 빈칸없이 제공됩니다.

> 사용법  
> 1) 위에서부터 내려오며 `____`만 채우세요.  
> 2) 각 섹션의 체크 테스트를 수행해보세요.  
> 3) 마지막 “실습: CIFAR-10 학습” 섹션을 실행해보세요(시간이 오래걸릴 수 있습니다.)


## 학습 목표

1. **Residual addition**(`out + identity`)이 왜 학습을 안정화하는지 설명할 수 있다.  
2. Shortcut의 **Option A(Identity + Zero-pad)** / **Option B(Projection)** 를 구분하고, 언제 필요한지(채널/stride 불일치) 판단할 수 있다.  
3. CIFAR-ResNet의 **stage 구조**와 “stage 시작에서만 downsample(stride=2)” 규칙을 코드로 구현할 수 있다.  
4. CIFAR 실험의 **depth = 6n + 2 규칙**을 코드로 계산하고 검증할 수 있다.  
5. 학습/평가 루프에서 `train() / eval() / no_grad()`의 역할을 코드 레벨에서 정확히 구분할 수 있다.

> 허브: [R01] [R02] [R04]


## Option A / Option B 용어 정리

CheatSheet 기준으로 **Option A는 “파라미터 없는 shortcut” 전체**를 의미합니다.

- **Option A (기본형 = Identity Shortcut)**  
  - 조건: `stride==1` **그리고** `in_channels == out_channels`  
  - 구현: `nn.Identity()`  
  - 의미: 입력을 **아무 변화 없이** 그대로 더할 수 있는 상황

- **Option A (응용형 = Zero-padding Shortcut, CIFAR 실험에서 사용)**  
  - 조건: stage 경계에서 `stride!=1` 이거나 `out_channels != in_channels`  
  - 구현: `ShortcutZeroPad`  
  - 의미: 학습 가능한 $W_s$ 없이 **(H,W)와 채널 수만 맞춰서** `F(x)+x`를 유지  
  - ⚠️ 주의: “항상 쓰는 identity”가 아니라, **차원이 안 맞을 때만** 등장하는 Option A 입니다.

- **Option B (Projection Shortcut)**  
  - 조건: 차원 불일치가 있을 때 “성능/안정성”을 위해 학습 가능한 매핑을 허용  
  - 구현: `ShortcutProjection` (보통 1×1 conv)

따라서 이 노트북에서 **`ShortcutZeroPad` 셀 제목에 Option A가 붙어 있어도**,  
그 의미는 “Option A 중에서도 **차원 불일치(mismatch) 상황을 해결하는 응용형**”을 구현한다는 뜻입니다.

> 허브: [R05] [R08]


## (선택) 실행을 위한 설치

이미 환경에 PyTorch/torchvision이 있다면 건너뛰세요.


In [ ]:

# 필요 시만 실행
# !pip -q install torch torchvision


## 0) Residual Learning 핵심 ↔ `BasicBlockV1.forward`의 "add" 라인

- **핵심 수식**:  \( y = F(x) + x \)  
- 직관: “원본(x)을 통째로 다시 만들지 말고, 필요한 수정분(F(x))만 더한다.”

> 허브: [R04] [R05]


## (준비) Import (모델 파트)


### 🔧 진단 헬퍼 (그대로 실행)

Check 셀이 실패할 때 **어느 빈칸을 봐야 하는지**와 **쿡북 어느 절로 가야 하는지**를 알려 줍니다.
정답은 알려주지 않습니다 — 방향만 줍니다.


In [ ]:
# =========================================================
# 진단 헬퍼 — Check 셀이 "틀렸다"가 아니라 "어디가 틀렸다"를 말하게 한다.
# 이 셀은 빈칸이 없습니다. 그대로 실행하세요.
# =========================================================

def check_shape(actual, expected, where):
    """shape을 비교하고, 어긋나면 어느 축이 왜 어긋났는지까지 짚어 준다.

    where: 어느 Check인지 ("zeropad" | "projection" | "block")
    """
    actual, expected = tuple(actual), tuple(expected)
    if actual == expected:
        print(f"[OK] {where} shape: {actual}")
        return

    N, C, H, W = 0, 1, 2, 3
    msg = [f"[X] {where}: 기대 {expected} / 실제 {actual}"]

    # 어느 축이 어긋났는지부터 분리한다 — 축마다 원인이 다르다
    if actual[C] != expected[C]:
        msg.append(f"  · 채널(축1)이 {actual[C]} ≠ {expected[C]}")
        if where == "zeropad":
            msg.append("    → 0 텐서를 이어 붙이는 부분을 보세요. 추가 채널 수 계산과")
            msg.append("      이어 붙이는 축이 맞는지. 쿡북 [C1-2]")
        elif where == "projection":
            msg.append("    → 1x1 conv의 out_channels 인자를 보세요. 쿡북 [C2-1]")
        else:
            msg.append("    → 블록 안 conv의 출력 채널, 또는 shortcut 분기를 보세요. 쿡북 [C3-5]")

    if actual[H] != expected[H] or actual[W] != expected[W]:
        msg.append(f"  · 공간 크기(축2,3)가 {actual[H]}x{actual[W]} ≠ {expected[H]}x{expected[W]}")
        if actual[H] > expected[H]:
            msg.append("    → 줄어들지 않았습니다. stride가 적용되는 자리를 빠뜨렸는지 보세요.")
        else:
            msg.append("    → 너무 많이 줄었습니다. stride가 두 번 먹었는지 보세요.")
        msg.append("      main path와 shortcut path가 **같은 배수로** 줄어야 합니다. 쿡북 [C1-1]")

    if actual[N] != expected[N]:
        msg.append(f"  · 배치(축0)가 {actual[N]} ≠ {expected[N]} — 배치 축을 건드렸습니다. 쿡북 [C1-1]")

    msg.append("  · 전체 대조표는 쿡북 [부록) 에러 메시지별 디버깅 표]")
    raise ValueError("\n".join(msg))


def check_close(name, got, want, tol=1e-5):
    """스칼라 값 비교용 (정확도 등)."""
    if abs(got - want) <= tol:
        print(f"[OK] {name}: {got}")
    else:
        raise ValueError(f"[X] {name}: 기대 {want} / 실제 {got}")


print("[준비] 진단 헬퍼 로드됨 — 이제 Check 셀이 어디가 틀렸는지 짚어 줍니다.")


In [ ]:
from typing import List, Type
import torch
import torch.nn as nn


## 1) Shortcut Option A (Identity + Zero-padding) ↔ `ShortcutZeroPad.forward`


✅ 핵심: **Option A는 (shape가 맞으면) Identity, (안 맞으면) Zero-padding** 으로 생각하면 가장 덜 헷갈립니다.
Option A는 **파라미터를 늘리지 않고**(학습되는 W_s 없음) 차원만 맞춥니다.

- stride로 H,W가 줄면 shortcut도 **같이 줄여야** 더하기가 가능합니다.
- 채널이 늘면 `0` 채널을 붙여서 `(N, out_channels, H, W)`로 맞춥니다.

> 허브: [R08] [R15]


In [ ]:
# =========================================================
# 1) Shortcut (Option A: zero-pad, Option B: projection)
# =========================================================

class ShortcutZeroPad(nn.Module):
    """
    Option A (CIFAR): **학습 파라미터 없이** 차원만 맞추는 shortcut.
    - stage 경계에서 (H,W)가 줄어들거나 채널이 늘어날 때도 'F(x) + shortcut(x)'가 가능하도록
      shortcut 경로의 shape를 맞춥니다.
    """

    def __init__(self, in_channels: int, out_channels: int, stride: int):
        super().__init__()
        if out_channels < in_channels:
            raise ValueError(
                "ShortcutZeroPad only supports channel increase (out_channels >= in_channels). "
                "Use projection shortcut (1x1 conv) for channel decrease."
            )
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.stride = stride

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # NOTE (Option A 정리)
        # - stride==1이고 채널 변화가 없으면, 사실상 identity와 동일하게 동작합니다.
        # - 차원이 불일치할 때만 downsample/zero-pad를 수행합니다.

        # [CS§1 Option A(Zero-padding)] (H,W) 맞추기: → [R08] [R15] [C1-1]
        # 힌트: 쿡북 [C1-1]  
        # 메인 경로가 stride로 해상도를 줄였다면, shortcut도 같은 비율로 줄여야 add가 가능합니다.
        # ================================================================================
        # 현재 Tensor의 shape은 (개수, 채널, 행, 열)입니다.
        # (A문장): 해상도를 줄였다면 개수와 채널은 그대로 두고 행과 열은 stride에 맞춰 줄이는 게 맞습니다.
        # (B문장): 예를 들어 tensor x의 형태가 (A, B)의 형태고 A는 그대로 B는 stride에 맞춰 해상도를 줄인다고 한다면 x = x[:, ::stirde]가 되어야 합니다.
        # A문장과 B문장을 연결해서 생각하면 빈칸을 채울 수 있습니다!
        # ================================================================================
        if self.stride != 1:
            x = _[_,_,____,____]  # [N-1] 설명: stride에 맞춰 (H,W)를 축소해 main path와 동일한 공간 크기로 맞춘다

        # [CS§1 Option A(Zero-padding)] (C) 맞추기: → [R05] [R08] [C1-2]
        # 출력 채널에 맞추기 위해 "추가로 필요한 채널 수"를 계산합니다.
        # ================================================================================
        # PyTorch에서 두 텐서를 더하려면 차원 크기가 완전히 같거나, 브로드캐스팅 규칙(크기가 1인 차원 확장)을 만족해야 합니다.
        # 하지만 이 코드에서 Main path를 통과한 F(x)는 채널이 out_channels로 늘어났지만, Shortcut의 x의 channel이 일치하지 않아 연산할 수 없는 경우가 있습니다.
        # 예를 들어 Shortcut의 x는 in_channels은 32로 (N, 32, H, W) F(x)의 out_channel은 64로 (N, 64, H, W)가 될 수 있습니다. 
        # 이 경우에는 채널 수가 서로 다르고 1도 아니기 때문에 연산 에러가 발생합니다.
        # 따라서 파라미터 추가 없이 연산 규칙을 만족시키기 위해, 부족한 채널 수만큼 0으로 채워진 텐서(zero matrix)를 만들어 채널 축에 이어 붙여 완전히 동일한 shape으로 만들어 줍니다.
        # ================================================================================
        pad_channels = ____ - ____  # [N-2] 설명: 출력 채널 수와 입력 채널 수의 차이(추가해야 하는 채널 개수)

        if pad_channels == 0:
            return x

        # [CS§1 Option A(Zero-padding)] 0 텐서 만들기: → [R05] [R08] [C1-2]
        # (N, 추가채널, H, W) 형태로 만들어 channel 축에 이어 붙일 준비를 합니다.
        zeros = torch.zeros(
            (x.size(0), ____, x.size(2), x.size(3)),  # [N-3] 설명: 두 번째 축이 "추가 채널" 크기
            device=x.device,
            dtype=x.dtype,
        )

        # [CS§1 Option A(Zero-padding)] channel 축으로 concat → [R05] [R08] [C1-2]
        return torch.cat([x, zeros], dim=____)  # [N-4] 설명: 채널 축으로 이어 붙여 out_channels를 맞춘다


### ✅ Check 1: ShortcutZeroPad shape 테스트
(빈칸을 채운 뒤 실행)

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.


In [ ]:
# ShortcutZeroPad: in=16 -> out=32, stride=2
sp = ShortcutZeroPad(in_channels=16, out_channels=32, stride=2)
x = torch.randn(4, 16, 32, 32)
y = sp(x)

check_shape(y.shape, (4, 32, 16, 16), "zeropad")


## 2) Shortcut Option B (Projection) ↔ `ShortcutProjection`

Option B는 1×1 conv(+BN)로 입력을 **학습적으로** 변환해 차원을 맞춥니다.

> 허브: [R05] [R08]


In [ ]:

class ShortcutProjection(nn.Module):
    """
    Option B: 1×1 conv(+BN)로 shortcut 경로도 학습 가능하게 만들어 차원을 맞춥니다.
    - 차원 불일치가 생기는 지점에서 W_s x 형태로 변환한 뒤 더할 수 있게 합니다.
    """

    def __init__(self, in_channels: int, out_channels: int, stride: int):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            stride=stride, 
            padding=0,
            bias=False,
        )
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # [CS§1 Projection Shortcut] → [R05] [R08] [C2-1]
        out = self.____(x)    # [N-5] 1x1 conv로 채널과 공간 크기를 맞춘다 → [R05] [R08] [C2-1] [CS§1]
        out = self.____(out)  # [N-6] BN을 적용한다 → [R09] [C2-1] [C3-2] [CS§1]
        return out


### ✅ Check 2: ShortcutProjection shape 테스트
(빈칸을 채운 뒤 실행)

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.


In [ ]:
sp = ShortcutProjection(in_channels=16, out_channels=32, stride=2)
x = torch.randn(4, 16, 32, 32)
y = sp(x)

check_shape(y.shape, (4, 32, 16, 16), "projection")


## 3) BasicBlockV1 (ResNet v1: post-activation) ↔ `BasicBlockV1`

ResNet v1의 기본 블록(논문 Figure 2)은 아래 흐름입니다.

1) conv → bn → relu  
2) conv → bn  
3) add(identity)  
4) relu


**CheatSheet 대응 포인트**
- shortcut 선택 로직: Identity / Zero-pad / Projection (Option A/B)
- Residual Addition (Eq.1) — 이 블록에서 **더해지는 두 텐서가 각각 무엇인지** 이름을 대 보세요.
  하나는 방금 만든 F(x)이고, 다른 하나는 위에서 미리 준비해 둔 것입니다.
- add 이후 ReLU: Post-activation (ResNet v1)

> 허브: [R06] [R07]


In [ ]:
# =========================================================
# 2) Residual Block (ResNet v1: post-activation)
# =========================================================

class BasicBlockV1(nn.Module):
    """
    Figure 2 (left): conv-bn-relu -> conv-bn -> add -> relu  (ResNet v1)
    """
    expansion = 1  # BasicBlock은 채널 확장(expansion)이 없습니다.

    def __init__(self, in_channels: int, out_channels: int, stride: int, shortcut: str = "projection"):
        super().__init__()

        # 메인 경로(F): 3×3 conv 두 번 (CIFAR-ResNet 기본)
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        # 두 번째 conv는 블록 내부에서 추가 downsample을 하지 않도록 stride=1 고정
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        # [CS§1 Shortcut 연결 방식] → [R05] [R07] [C3-5]
        # (1) shape가 같으면 가장 단순한 shortcut(항등)을 쓴다.
        if stride == 1 and in_channels == out_channels:
            self.shortcut = nn.____()  # [N-7] 설명: 입력을 그대로 통과시키는 shortcut 모듈
        # (2) shape가 다르면: Option A(ZeroPad) 또는 Option B(Projection) 중 선택
        else:
            if shortcut == "projection":
                self.shortcut = ____(in_channels, out_channels, stride=stride)  # [N-8] 설명: 학습 가능한 1×1 conv shortcut
            elif shortcut == "zero_pad":
                self.shortcut = ____(in_channels, out_channels, stride=stride)  # [N-9] 설명: 파라미터 없이 0 채널을 붙이는 shortcut
            else:
                raise ValueError(f"Unknown shortcut mode: {shortcut}")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # [CS§1 Shortcut 연결 방식] shortcut 경로 텐서 준비 → [R05] [C3-4]
        identity = ____.____(____)  # [N-10] 역할: 메인 경로와 더할 수 있도록 bypass 텐서를 만든다

        # [CS§0 Residual Function] main path: conv1 -> bn1 -> relu → [R04] [R09] [C3-1] [C3-2]
        out = self.____(x)    # [N-11] main path 첫 conv → [R04] [C3-1]
        out = self.____(out)  # [N-12] BN (conv 바로 뒤, activation 앞) → [R09] [C3-2]
        out = self.____(out)  # [N-13] ReLU → [R06] [C3-3]

        # [CS§0 Residual Function] main path: conv2 -> bn2 (relu 없음) → [R04] [R06] [R09] [C3-1] [C3-2]
        out = self.____(out)  # [N-14] main path 둘째 conv → [R04] [C3-1]
        out = self.____(out)  # [N-15] BN. 여기까지가 F(x) → [R09] [C3-2]

        # [CS§0 Residual Addition (Eq.1)] Residual addition → [R04] [R05] [C3-4]
        out = ____ + ____  # [N-16] 역할: residual branch 결과와 shortcut 결과를 결합한다

        # [CS§2 Post-activation] Post-activation (ResNet v1) → [R06] [C3-3]
        out = ____.____(out)  # [N-17] 역할: 합산 결과에 활성화를 적용해 블록 출력을 만든다
        return out


### ✅ Check 3: BasicBlockV1 shape 테스트
(빈칸을 채운 뒤 실행)

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.


In [ ]:
blk = BasicBlockV1(
    in_channels=16,
    out_channels=32,
    stride=2,
    shortcut="projection"
)

x = torch.randn(2, 16, 32, 32)
y = blk(x)

check_shape(y.shape, (2, 32, 16, 16), "block")


## 4) (심화) BottleneckV1 ↔ `BottleneckV1`

깊은 ResNet(50/101/152 등)에서 중요한 구조입니다.

- 1×1로 채널을 줄이고 → 3×3에서 계산 → 1×1로 채널을 다시 확장(expansion=4)
- 연산량(특히 3×3)을 줄이면서 깊이를 늘립니다.


> 허브: [R12]

> 참고 구현은 `노트북_정답.ipynb`로 옮겼습니다 — 3) BasicBlockV1 구간(N-6~N-13)의 답이 그대로 드러나기 때문입니다.
> 개념은 쿡북 `[4) BottleneckV1]`을 보세요.


### ✅ Check 4: BottleneckV1 shape 테스트 (심화, 선택)

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.


In [ ]:
# Check 4는 심화(선택)입니다. BottleneckV1은 이 노트북에 정의돼 있지 않습니다
# (3) BasicBlockV1 구간의 답이 드러나서 정답 노트북으로 옮겼습니다).
# 정의가 없으면 조용히 건너뜁니다 — 이 셀이 빨간불이어도 N-1~N-26과는 무관합니다.

if "BottleneckV1" not in dir():
    print("[SKIP] BottleneckV1이 정의되지 않았습니다. 심화 항목이라 건너뜁니다.")
    print("       직접 해보려면 쿡북 [4) BottleneckV1]을 보고 구현한 뒤 이 셀을 다시 실행하세요.")
else:
    blk = BottleneckV1(in_channels=64, planes=16, stride=2, shortcut="projection")
    x = torch.randn(2, 64, 32, 32)
    y = blk(x)
    # out_channels = planes*expansion = 16*4=64, stride=2 => 16x16
    assert y.shape == (2, 64, 16, 16), f"shape mismatch: {y.shape}"
    print("[OK] BottleneckV1 shape:", y.shape)


## 5) ResNetV1 전체 구조 ↔ `ResNetV1.__init__`, `_make_layer`, `forward`

### 핵심 규칙 (CheatSheet 좌표)
- Stage 구조: CheatSheet §4
- Downsampling at Stage Start: CheatSheet §4
- Small Stem (CIFAR): CheatSheet §5
- Global Average Pooling: CheatSheet §6

### 핵심 규칙
- **Stage 구조**: stage 내부에서는 (H,W,채널 패턴)이 일정  
- **Downsampling은 stage 시작에서만**(첫 블록 stride=2)  
- CIFAR는 입력이 작아 **small stem(3×3, stride=1)**로 시작  
- 마지막은 **Global Average Pooling(GAP)** 후 FC

> 허브: [R15] [R16] [R17]


In [ ]:

class ResNetV1(nn.Module):
    def __init__(
        self,
        block: Type[nn.Module],
        layers: List[int],
        num_classes: int,
        stem: str,        # "imagenet" or "cifar"
        shortcut: str,    # "projection" or "zero_pad"
        cifar_base_channels: int = 16,
    ):
        super().__init__()
        self.shortcut = shortcut

        # [CS§5 Small Stem] Stem 구성 (CIFAR vs ImageNet) → [R15] [C5-1]
        if stem == "imagenet":
            self.in_channels = 64
            self.stem = nn.Sequential(
                nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            )
            stage_planes = [64, 128, 256, 512]
        elif stem == "cifar":
            # CIFAR는 입력이 작으므로 3×3 conv로 “small stem” (해상도 유지)
            self.in_channels = cifar_base_channels
            self.stem = nn.Sequential(
                nn.Conv2d(3, cifar_base_channels, kernel_size=3, stride=1, padding=1, bias=False),
                nn.BatchNorm2d(cifar_base_channels),
                nn.ReLU(inplace=True),
            )
            # base 채널에서 stage마다 2배씩 증가 (16 → 32 → 64)
            # [CS§4 Stage-based Design] stage 채널 스케줄 → [R16] [C5-4]
            stage_planes = [cifar_base_channels, cifar_base_channels * ____, cifar_base_channels * ____]  # [N-18] → [R16] [C5-4] [CS§4]
        else:
            raise ValueError("stem must be 'imagenet' or 'cifar'")

        # [CS§4 Downsampling at Stage Start] → [R16] [C5-4]
        # stage 시작에서만 downsample (CIFAR: layer2, layer3 진입 시 stride=2)
        self.layer1 = self._make_layer(block, stage_planes[0], layers[0], stride=____)  # [N-19] → [R16] [C5-4] [CS§4]
        self.layer2 = self._make_layer(block, stage_planes[1], layers[1], stride=____)  # [N-20] → [R16] [C5-4] [CS§4]
        self.layer3 = self._make_layer(block, stage_planes[2], layers[2], stride=____)  # [N-21] → [R16] [C5-4] [CS§4]

        if stem == "imagenet":
            self.layer4 = self._make_layer(block, stage_planes[3], layers[3], stride=2)
            final_ch = stage_planes[3] * getattr(block, "expansion", 1)
        else:
            self.layer4 = None
            final_ch = stage_planes[2] * getattr(block, "expansion", 1)

        # [CS§6 Global Average Pooling] Global Average Pooling + FC → [R17] [C5-3]
        self.avgpool = nn.AdaptiveAvgPool2d((____, ____))  # [N-22] → [R17] [C5-3] [CS§6]
        self.fc = nn.Linear(final_ch, num_classes)

    def _make_layer(self, block: Type[nn.Module], planes: int, blocks: int, stride: int) -> nn.Sequential:
        """
        stage를 조립합니다.
        - 첫 블록만 stride를 적용해 downsample 가능
        - 이후 블록들은 stride=1로 유지
        """
        layers = []
        expansion = getattr(block, "expansion", 1)

        # Bottleneck은 (in_channels, planes, ...) 입력을 기대 (planes=내부 채널)
        # expansion != 1 이면 bottleneck 계열이다 (BasicBlock은 expansion == 1)
        if expansion != 1:
            # 아래 BasicBlock 분기의 설명 블록과 논리가 같습니다 (첫 블록만 채널이 다르다).
            # 다만 bottleneck은 출력 채널이 planes가 아니라 planes를 expansion배 한 값입니다. [C4-4]
            layers.append(block(self.in_channels, planes, stride=stride, shortcut=self.shortcut))
            self.in_channels = ____ * ____  # [N-23] 설명: bottleneck 출력 채널로 in_channels를 갱신한다 → [R12] [R16] [C4-4]
            for _ in range( ____ , blocks):  # [N-24] 설명: 첫 블록을 이미 추가했으니, 나머지 블록만 반복한다 → [R16] [C5-2]
                layers.append(block(self.in_channels, planes, stride=1, shortcut=self.shortcut))

        # BasicBlock은 (in_channels, out_channels, ...) 입력을 기대
        else:
            # ================================================================================
            # stage 하나는 블록 여러 개를 이어 붙인 것입니다. 그런데 첫 블록과 나머지 블록은
            # 받는 채널 수가 다릅니다.
            #
            # 예: stage2는 16채널을 받아 32채널을 내보냅니다. 블록이 3개라면
            #     첫 블록  : 16 → 32   (stride=2, downsample은 여기서만)
            #     둘째 블록: 32 → 32
            #     셋째 블록: 32 → 32
            #
            # 첫 블록만 16을 받고 그 뒤로는 전부 32를 받습니다. 그래서 첫 블록을 넣은 뒤에
            # "다음 블록이 받을 채널 수"를 갱신해 두지 않으면, 둘째 블록이 여전히 16을 기대한
            # 채로 만들어져 실행 시점에 shape mismatch가 납니다.
            # (에러 메시지는 쿡북 [부록] 표의 "expected input[..] to have B channels" 줄)
            # ================================================================================
            out_channels = ____ * ____  # [N-25] 설명: stage의 출력 채널 수(planes와 expansion의 관계를 생각) → [R12] [R16] [C4-4]
            layers.append(block(self.in_channels, out_channels, stride=stride, shortcut=self.shortcut))
            self.in_channels = ____  # [N-26] 설명: 다음 블록 입력 채널을 현재 stage 출력 채널로 갱신한다 → [R16] [C5-2]

            # 위에서 첫 블록을 이미 append 했습니다. 그러니 이 루프가 blocks번 돌면
            # 블록이 blocks+1개가 됩니다 — Check 4.5가 개수를 세서 잡아 줍니다.
            for _ in range(____, blocks):  # [N-27] 설명: 첫 블록은 이미 넣었으니 남은 블록들만 추가한다 → [R16] [C5-2]
                layers.append(block(self.in_channels, out_channels, stride=1, shortcut=self.shortcut))

        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        if self.layer4 is not None:
            x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


# =========================================================
# 4) Factory (CIFAR: depth = 6n + 2)
# =========================================================


### ✅ Check 4.5: `_make_layer`만 따로 확인 (중간 점검)

위 셀은 빈칸이 10개라 **전부 맞을 때까지 아무 피드백이 없습니다.**
이 셀은 `_make_layer` 부분(N-23~N-27)만 떼어 먼저 확인합니다 — 나머지가 아직 틀려도 됩니다.

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.


In [ ]:
# _make_layer만 확인: stage 하나를 만들어 채널과 크기가 맞는지 본다.
# ResNetV1 전체가 완성되지 않아도 이 부분만 맞으면 통과합니다.

_m = ResNetV1.__new__(ResNetV1)      # __init__을 건너뛰고 껍데기만 만든다
nn.Module.__init__(_m)
_m.in_channels = 16
_m.shortcut = "projection"

_stage = _m._make_layer(BasicBlockV1, planes=32, blocks=3, stride=2)
_out = _stage(torch.randn(2, 16, 32, 32))

check_shape(_out.shape, (2, 32, 16, 16), "block")

if len(_stage) != 3:
    raise ValueError(
        f"[X] stage에 블록이 {len(_stage)}개입니다 (3개여야 정상).\n"
        "  → 반복 횟수를 보세요. 첫 블록은 루프 **밖에서** 이미 추가했으니\n"
        "     루프는 나머지만 돌아야 합니다. 쿡북 [C5-2]"
    )
print("[OK] _make_layer: 블록 3개, 출력", tuple(_out.shape))


## 6) CIFAR Factory: depth = 6n + 2 ↔ `resnet_cifar`

CIFAR-ResNet에서 depth는 **6n+2** 형태여야 합니다.

- conv는 블록당 2개  
- stage가 3개이고 각 stage에 n 블록  
- 그래서 conv 층 수가 2×(3×n)=6n, 여기에 stem conv 1개 + classifier 1개를 더해 6n+2

> 허브: [R15]


In [ ]:

def resnet_cifar(depth: int, num_classes: int = 10, shortcut: str = "zero_pad"):
    """
    CIFAR-ResNet (논문 §4.2):
    - depth = 6n + 2 형태여야 함 (예: 20/32/44/56/110/...)
    - 각 stage의 블록 개수는 모두 n
    """
    # (1) depth 규칙 검사
    if (____ - ____) % ____ != ____:  # [N-28] 설명: depth가 6n+2 형태가 아니면 예외를 발생시킨다 → [R15] [C6-2] [CS§5]
        raise ValueError("CIFAR depth must be 6n+2")

    # (2) n 역산
    n = (____ - ____) // ____  # [N-29] 설명: depth에서 n을 역으로 계산한다 (정수로 떨어져야 함) → [R15] [C6-1] [CS§5]

    # (3) stage별 블록 개수: [n, n, n]
    return ResNetV1(BasicBlockV1, [____, ____, ____], num_classes, stem="cifar", shortcut=shortcut)  # [N-30] → [R15] [C6-3] [CS§5]


## Appendix: ImageNet용 ResNet 팩토리 함수

CIFAR-10 실험은 논문 §4.2의 **단순화된 구조(6n+2, 32×32 입력, 3 stages)**를 쓰지만,
ResNet의 “근본”은 ImageNet 설정(224×224 입력, 4 stages, 18/34는 BasicBlock, 50+는 Bottleneck)입니다.

- 이 섹션은 **학습 필수는 아니고**, 구조 감 잡기/확장 실습을 위해 **참고로 포함**했습니다.
- 빈칸은 없고, 그대로 실행 가능한 유틸 함수입니다.

> 허브: [R12] [R16]


In [ ]:

# ResNet 모델 생성 함수 (ImageNet 용) — 참고용
def resnet_imagenet(depth: int, num_classes: int = 1000, shortcut: str = "projection"):
    """
    ImageNet 설정의 ResNet 팩토리 함수.
    - 18/34: BasicBlock
    - 50/101/152: Bottleneck
    - shortcut: "projection" 또는 "zero_pad"(실험/학습용). 실전 ImageNet은 보통 projection을 사용합니다.
    """
    if depth == 18:
        return ResNetV1(BasicBlockV1, [2, 2, 2, 2], num_classes, stem="imagenet", shortcut=shortcut)
    if depth == 34:
        return ResNetV1(BasicBlockV1, [3, 4, 6, 3], num_classes, stem="imagenet", shortcut=shortcut)
    if depth == 50:
        return ResNetV1(BottleneckV1, [3, 4, 6, 3], num_classes, stem="imagenet", shortcut=shortcut)
    if depth == 101:
        return ResNetV1(BottleneckV1, [3, 4, 23, 3], num_classes, stem="imagenet", shortcut=shortcut)
    if depth == 152:
        return ResNetV1(BottleneckV1, [3, 8, 36, 3], num_classes, stem="imagenet", shortcut=shortcut)
    raise ValueError("Unsupported depth for ImageNet ResNet")


### ✅ Check 5: ResNet forward / depth rule 테스트

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.


In [ ]:

# depth rule
try:
    resnet_cifar(depth=20)  # 20=6*3+2 OK
except Exception as e:
    raise AssertionError(f"depth=20 should be valid, but got: {e}")

try:
    resnet_cifar(depth=21)  # invalid
    raise AssertionError("depth=21 should be invalid but passed")
except ValueError:
    print("[OK] invalid depth correctly raises ValueError")

# forward shape
m = resnet_cifar(depth=56, num_classes=10, shortcut="zero_pad")
x = torch.randn(4, 3, 32, 32)
with torch.no_grad():
    out = m(x)

expected = (4, 10)
actual = tuple(out.shape)
if actual != expected:
    raise ValueError(f"output shape mismatch: expected {expected}, got {actual}")

print("[OK] ResNet forward:", out.shape)



## 7) Train/Eval 루프 ↔ `train_one_epoch`, `evaluate`

### 핵심 포인트
- 학습: `model.train()` + gradient 필요  
- 평가: `model.eval()` + `torch.no_grad()`로 gradient off

> 허브: [R09] [R13]


In [ ]:

# =========================================================
# 7) Train/Eval 유틸리티 (빈칸 없음)
# - 아래 실습 섹션(§8)에서도 그대로 활용할 수 있습니다.
# =========================================================

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T


def build_transforms():
    """논문 §4.2 CIFAR-10 augmentation (기본 설정)."""
    train_tf = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
    ])
    test_tf = T.Compose([T.ToTensor()])
    return train_tf, test_tf


def build_dataloaders(batch_size: int = 128, num_workers: int = 4):
    train_tf, test_tf = build_transforms()
    trainset = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
    testset = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)

    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return trainloader, testloader


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)

    return total_loss / max(total, 1), correct / max(total, 1)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item() * x.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    return total_loss / max(total, 1), correct / max(total, 1)


### ✅ Check 6: (데이터 다운로드 없이) 더미 배치로 train/eval 루프 스모크 테스트

**2부(과적합 검사)는 30초쯤 걸립니다.** shape이 맞는데도 `residual` 경로가 끊긴 경우를 잡습니다 —
쿡북 부록이 "잡기 어렵다"고 경고하는 바로 그 부류입니다.

> **실행 전에 한 번**: 출력 shape이 얼마일지 먼저 적어 보세요 — `(  ,  ,  ,  )`.
> 틀렸다면 어디서 어긋났는지가 곧 배울 지점입니다.


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 1부: 루프가 도는가 (스모크) --------------------------------
model = resnet_cifar(depth=20, num_classes=10, shortcut="projection").to(device)
x = torch.randn(64, 3, 32, 32)
y = torch.randint(0, 10, (64,))
loader = DataLoader(TensorDataset(x, y), batch_size=16, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

train_loss, train_acc = train_one_epoch(model, loader, criterion, optimizer, device)
test_loss, test_acc = evaluate(model, loader, criterion, device)

for nm, v in [("train_loss", train_loss), ("train_acc", train_acc),
              ("test_loss", test_loss), ("test_acc", test_acc)]:
    if not isinstance(v, float):
        raise TypeError(f"{nm}는 float여야 합니다. 실제: {type(v)}")
print(f"[OK] 1부 루프가 돕니다: train_acc={train_acc:.3f} test_acc={test_acc:.3f}")

# --- 2부: 모델이 정말 학습되는가 (과적합 검사) --------------------
# 왜 필요한가: 1부는 타입만 봅니다. 그래서 아래 버그를 전부 통과시킵니다.
#   · 덧셈 뒤 ReLU 누락      · shortcut을 더하지 않음
#   · shortcut 경로에 ReLU   · BN을 conv 뒤에 안 붙임
# 이런 버그는 shape이 맞아서 에러가 안 나고 "정확도만 조용히" 낮아집니다.
#
# 왜 depth=56인가: 얕은 net은 shortcut이 없어도 32개쯤은 외웁니다(실측 depth=20에서 100%).
# 깊어야 갈립니다 — 실측 depth=56에서 residual 100% vs shortcut 없음 44%.
# 이것이 논문 [R02]가 말하는 degradation을 아주 작게 재현한 것이기도 합니다.

torch.manual_seed(0)
tiny_x = torch.randn(32, 3, 32, 32)
tiny_y = torch.randint(0, 10, (32,))
tiny = DataLoader(TensorDataset(tiny_x, tiny_y), batch_size=32, shuffle=False)

m2 = resnet_cifar(depth=56, num_classes=10, shortcut="projection").to(device)
opt2 = optim.SGD(m2.parameters(), lr=0.05, momentum=0.9)

acc = 0.0
for _ in range(40):
    _, acc = train_one_epoch(m2, tiny, criterion, opt2, device)

if acc < 0.9:
    raise ValueError(
        f"[X] 32개짜리 작은 데이터를 40번 반복했는데 train_acc={acc:.2f}입니다 (0.90 이상이어야 정상).\n"
        "  shape은 맞는데 **의미**가 틀렸을 때 나오는 증상입니다. 아래를 순서대로 보세요:\n"
        "  1) 덧셈 뒤 ReLU를 넣었는가 (블록 마지막) — 쿡북 [C3-3]\n"
        "  2) shortcut 결과를 실제로 **더했는가** — 쿡북 [C3-4]\n"
        "  3) shortcut 경로에 ReLU를 넣지는 않았는가 (identity 통로가 막힌다) — 쿡북 [C3-5]\n"
        "  4) BN을 conv 바로 뒤에 붙였는가 — 쿡북 [C3-2]\n"
        "  5) 결과 변수를 덮어쓰는 사슬이 끊기지 않았는가 — 쿡북 [C0]"
    )

print(f"[OK] 2부 학습됩니다: 작은 데이터 train_acc={acc:.2f}")
print("[OK] Check 6 통과 — 루프도 돌고 residual 경로도 살아 있습니다.")


## 8) 실습: CIFAR-10 학습을 실제로 돌려보기 (논문 설정 정렬)

아래 코드는 **논문 §4.2 설정에 맞춘 CIFAR-10 학습 스켈레톤**입니다.

- 이 섹션은 **빈칸이 없습니다.**
- `epochs=164`는 시간이 오래 걸릴 수 있으니, 처음에는 `epochs=1~3`으로 줄여서 동작 확인을 권장합니다.
- 노트북에서는 `resnet_cifar`가 이미 위에서 정의되어 있으므로, `from model import ...` 같은 외부 import가 필요 없습니다.

> 허브: [R13] [R11] [R14]


In [ ]:

# train_cifar10.py (paper-aligned CIFAR-10 training skeleton) — notebook friendly

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T


def main(epochs: int = 164, batch_size: int = 128, num_workers: int = 4):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Examples: 20/32/44/56/110... where depth = 6n+2
    model = resnet_cifar(depth=56, num_classes=10, shortcut="zero_pad").to(device)

    # Data augmentation described in the paper (§4.2)
    train_tf = T.Compose([
        T.RandomCrop(32, padding=4),
        T.RandomHorizontalFlip(),
        T.ToTensor(),
    ])
    test_tf = T.Compose([T.ToTensor()])

    trainset = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=train_tf)
    testset = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)

    trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)

    # Paper: divide by 10 at 32k and 48k iters, stop at 64k iters.
    # 50k/128 ≈ 391 iters/epoch → 32k≈82ep, 48k≈123ep, 64k≈164ep
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[82, 123], gamma=0.1)

    for epoch in range(epochs):
        model.train()
        for x, y in trainloader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        model.eval()
        correct, total, test_loss = 0, 0, 0.0
        with torch.no_grad():
            for x, y in testloader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                test_loss += criterion(logits, y).item() * x.size(0)
                pred = logits.argmax(dim=1)
                correct += (pred == y).sum().item()
                total += y.size(0)

        scheduler.step()
        print(f"epoch={epoch:03d} acc={100.0*correct/total:.2f} loss={test_loss/total:.4f}")


# ✅ 실행 예시 (처음엔 epochs를 줄여서!)
# main(epochs=3)


## 9) (실험) plain vs residual — degradation 직접 재현

논문 Figure 1이 보여주는 것을 **직접 돌려 확인**한다. 깊은 plain net이 얕은 것보다 **training error가 높다**는 것,
그리고 shortcut을 넣으면 그 역전이 사라진다는 것.

이 셀은 학습자 빈칸이 아니다 — 그대로 실행만 하면 된다. 3 epoch면 충분하고, GPU 없이 MPS/CPU로도 돌아간다.

> 허브: [R02] [R10] [R11]

**주의**: 여기서 쓰는 `PlainBlock`은 이 셀 안에서만 정의한다. 위의 `BasicBlockV1`(빈칸)은 건드리지 않는다 —
덧셈을 빼려면 `forward`를 고쳐야 하는데 그 `forward`가 곧 N-9~N-13 빈칸이기 때문이다.


### ✋ 실행 전에 먼저 예측하세요

**돌리기 전에** 아래 표를 채웁니다. 논문 [R02]와 [R11]을 읽었다면 방향은 예측할 수 있습니다.
맞히는 게 목적이 아니라 **자기 예측과 결과를 견주는 것**이 목적입니다.

`>` `<` `≈` 중 하나로 적으세요 (training error 기준).

| 예측 | 20층 vs 44층 | 근거로 삼은 R |
| --- | --- | --- |
| plain | 20층 ___ 44층 | |
| residual | 20층 ___ 44층 | |

그리고 한 줄 더: **plain에서 깊은 쪽이 더 나쁘다면, 그것이 overfitting이 아니라는 걸
무엇으로 알 수 있나요?** (힌트: 지금 재는 것이 train인지 test인지)

```
답: 
```

적었으면 아래 셀을 실행합니다.


In [ ]:
# =========================================================
# 9) plain vs residual — degradation 재현 (실행만 하면 됨)
# =========================================================
import torch, torch.nn as nn, torchvision, torchvision.transforms as T

DEPTHS = (20, 44)   # 논문 §4.2의 n={3,7}
EPOCHS = 3          # 3 epoch면 역전이 이미 보인다
dev = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")


class PlainBlock(nn.Module):
    """이 셀 전용. residual=False면 shortcut 덧셈을 아예 하지 않는다."""
    def __init__(self, cin, cout, stride, residual):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.b2 = nn.BatchNorm2d(cout)
        self.relu = nn.ReLU(inplace=True)
        self.residual, self.stride, self.cin, self.cout = residual, stride, cin, cout

    def _shortcut(self, x):                      # Option A (zero-pad)
        if self.stride == 1 and self.cin == self.cout:
            return x
        s = self.stride
        x = torch.nn.functional.avg_pool2d(x, 1, stride=s)   # 위 빈칸의 슬라이싱과 같은 효과
        pad = torch.zeros(x.size(0), self.cout - self.cin, x.size(2), x.size(3), device=x.device)
        return torch.cat([x, pad], 1)

    def forward(self, x):
        out = self.relu(self.b1(self.c1(x)))
        out = self.b2(self.c2(out))
        if self.residual:
            out = out + self._shortcut(x)
        return self.relu(out)


def build(depth, residual):
    n = divmod(depth - 2, 6)[0]        # 6n+2 규칙 — 유도는 N-24/N-25에서 직접 하세요
    layers = [nn.Conv2d(3, 16, 3, 1, 1, bias=False), nn.BatchNorm2d(16), nn.ReLU(True)]
    cin = 16
    for planes, stride in [(16, 1), (32, 2), (64, 2)]:
        for i in range(n):
            layers.append(PlainBlock(cin, planes, stride if i == 0 else 1, residual)); cin = planes
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10)]
    return nn.Sequential(*layers)


tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(), T.ToTensor()])
ds = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=tf)
dl = torch.utils.data.DataLoader(ds, batch_size=128, shuffle=True, num_workers=0, drop_last=True)

results = {}
for residual in (False, True):
    for depth in DEPTHS:
        torch.manual_seed(0)
        model = build(depth, residual).to(dev)
        opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
        crit = nn.CrossEntropyLoss()
        for ep in range(EPOCHS):
            model.train(); correct = total = 0
            for x, y in dl:
                x, y = x.to(dev), y.to(dev)
                opt.zero_grad(); out = model(x); loss = crit(out, y); loss.backward(); opt.step()
                correct += (out.argmax(1) == y).sum().item(); total += y.numel()
            err = 100 - 100 * correct / total
        results[("residual" if residual else "plain", depth)] = err
        print(f"{'residual' if residual else 'plain':>8}-{depth}: train error {err:.2f}%")

print()
print("plain :", f"{results[('plain', DEPTHS[0])]:.2f}% (20층) vs {results[('plain', DEPTHS[1])]:.2f}% (44층)")
print("resid.:", f"{results[('residual', DEPTHS[0])]:.2f}% (20층) vs {results[('residual', DEPTHS[1])]:.2f}% (44층)")
print()
print("plain에서 깊은 쪽이 더 나쁘면 degradation이 재현된 것이다 —")
print("데이터를 덜 본 것도, gradient가 사라진 것도 아닌데 training error가 높다.")


### 실행 결과 예시 — **자기 결과를 본 뒤에** 펼치세요

<details>
<summary>이전 실행 결과 (Mac MPS, 3 epoch, seed 0) — 클릭해서 펼치기</summary>


| | 20층 | 44층 | 판정 |
| --- | --- | --- | --- |
| **plain** | 48.44% | **71.39%** | 깊은 쪽이 **23pp 더 나쁨** → degradation |
| **residual** | 37.60% | 41.80% | 역전 사라짐 |

읽는 법: plain 행에서 44층이 20층보다 training error가 높다. 이것이 [R02]가 말하는 degradation이고,
**overfitting이 아니다** — test가 아니라 train에서 지고 있기 때문이다.
residual 행에서 그 역전이 완화되는 것이 [R11]의 결과다.

> 3 epoch는 논문 예산(64k iteration)의 극히 일부라 residual도 아직 20층이 낮다.
> 여기서 볼 것은 절대 수치가 아니라 **plain에서만 격차가 크게 벌어진다**는 사실이다.

</details>
